# **Agent Middlewares**

## **What's Covered?**
1. abc
2. xyz

## **Introduction to Middleware**

### **What is Middleware?**
Middleware is a feature in LangChain that lets you step inside an agent’s workflow and make changes before, during, or after the model does something.

Think of it like a checkpoint system where you can plug in extra logic — without modifying the main agent code.

### **What You Can Do With Middleware?**
Here are the major ways middleware helps, with simple explanations:
1. Change the Input Before the Model Thinks
    - Removing unnecessary chat history using message trimming or by summarising messages
    - Add missing context using vector db connections
    - Injecting system messages
    - Enforcing prompt templates
2. Check or Fix the Model’s Output
    - Detect hallucinations
    - Apply safety filters (PII check, bad words removal)
    - Reformat the output (JSON shaping, title casing)
    - Block content that violates policy
3. Control Tool Execution
    - Retry a tool call if it fails
    - Stop execution early if the answer is already found
    - Choose different tools based on context
    - Replace a failing tool with a backup option
4. Track and Monitor Behavior
    - Logs
    - Debugging
    - Analytics dashboards
    - Production monitoring
    - Token usage analysis
5. Apply Guardrails and Operational Controls
    - Rate limiting per user
    - Preventing sensitive information leaks
    - Stopping infinite agent loops
    - Blocking large prompt injections
6. Switch or Customize the Model on the Fly
    - Query complexity
    - Cost budget
    - User tier
    - Context type

### **Why Middleware Is Valuable?**
Middleware gives you more control, flexibility, and safety over how your AI agent behaves. It allows you to adjust how the system works depending on your product needs.

Middleware gives you all this **without rewriting the agent logic**.

You attach helper logic from the outside, and it automatically hooks into the agent’s execution pipeline.

That means:
- Faster experimentation
- Cleaner code
- Less duplication
- Easier debugging
- Safer deployments

### **Understanding this with Agent Loop**
The core agent loop involves:
1. calling a model,
2. letting it choose tools to execute, and
3. then finishing when it calls no more tools

Middleware exposes hooks before and after each of those steps.

<div style="display:flex; gap:20px;">
    <img src="assets/agent_loop.png" style="width:40%; height:auto;">
    <img src="assets/agent_loop_with_middlewares.png" style="width:40%; height:auto;">
</div>

### **Built-in Middleware (Provider Agnostic)**

LangChain provides prebuilt middleware for common use cases. Each middleware is production-ready and configurable for your specific needs. The following middleware work with any LLM provider:
| Middleware            | Description                                                                 |
|-----------------------|-----------------------------------------------------------------------------|
| Summarization         | Automatically summarize conversation history when approaching token limits. |
| Human-in-the-loop     | Pause execution for human approval of tool calls.                            |
| Model call limit      | Limit the number of model calls to prevent excessive costs.                  |
| Tool call limit       | Control tool execution by limiting call counts.                              |
| Model fallback        | Automatically fallback to alternative models when the primary fails.        |
| PII detection         | Detect and handle Personally Identifiable Information (PII).                |
| To-do list            | Equip agents with task planning and tracking capabilities.                  |
| LLM tool selector     | Use an LLM to select relevant tools before calling the main model.           |
| Tool retry            | Automatically retry failed tool calls with exponential backoff.             |
| Model retry           | Automatically retry failed model calls with exponential backoff.            |
| LLM tool emulator     | Emulate tool execution using an LLM for testing purposes.                   |
| Context editing       | Manage conversation context by trimming or clearing tool uses.              |
| Shell tool            | Expose a persistent shell session to agents for command execution.           |
| File search           | Provide Glob and Grep search tools over filesystem files.                    |


### **Implementing Middlewares**

Add middleware by passing them to `create_agent`:
```python
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware, HumanInTheLoopMiddleware

agent = create_agent(
    model="gpt-4o",
    tools=[...],
    middleware=[
        SummarizationMiddleware(...),
        HumanInTheLoopMiddleware(...)
    ],
)
```

Reference Docs:
- https://docs.langchain.com/oss/python/langchain/middleware/built-in
- https://docs.langchain.com/oss/python/langchain/guardrails

Reference Video:
- https://academy.langchain.com/courses/take/langchain-essentials-python/lessons/69388329-lesson-6-memory
- https://academy.langchain.com/courses/take/foundation-introduction-to-langchain-python/lessons/71234843-course-overview

### **Middleware**

https://docs.langchain.com/oss/python/langchain/middleware/overview

https://docs.langchain.com/oss/python/langchain/middleware/built-in

https://docs.langchain.com/oss/python/langchain/middleware/custom

https://docs.langchain.com/oss/python/langchain/runtime

Test:
https://docs.langchain.com/oss/python/langchain/test

API Reference:  
https://reference.langchain.com/python/langchain/middleware/?_gl=1*1akd7zx*_gcl_au*ODYzMzY4ODc1LjE3NjM4MjAwNjA.*_ga*MTk3NjE3OTA4MC4xNzIzMDI3MjE3*_ga_47WX3HKKY2*czE3NjU5MDcxNjIkbzY2JGcxJHQxNzY1OTA3MzExJGo2MCRsMCRoMA..

### **Define Agent State with middleware**

Use middleware to define custom state when your custom state needs to be accessed by specific middleware hooks and tools attached to said middleware.

https://docs.langchain.com/oss/python/langchain/agents#memory

### **Context Engineering (Model, Tool and Lifecycle)**

LangChain middleware is the mechanism under the hood that makes context engineering practical for developers using LangChain.

https://docs.langchain.com/oss/python/langchain/context-engineering


## **Dynamic Model Selection and Prompt Selection**

@wrap_model_call:
- https://docs.langchain.com/oss/python/langchain/agents#dynamic-model

@dynamic_prompt:
- https://docs.langchain.com/oss/python/langchain/agents#dynamic-system-prompt

## **Tool Error Handling**

@wrap_tool_call:
- https://docs.langchain.com/oss/python/langchain/agents#tool-error-handling



## Step 4: Built-in Middleware (Prebuilt Superpowers)

**Add via `middleware=[]` list. Stack them.**

```python
from langchain.agents.middleware import (
    summarization_middleware,
    human_in_the_loop_middleware,
    pii_redaction_middleware
)

agent = create_agent(
    model=model,
    tools=[get_weather],
    middleware=[
        summarization_middleware(model="gpt-4o-mini", trigger={"tokens": 500}),  # Auto-summarize history
        human_in_the_loop_middleware(interrupt_on={"get_weather": {"allowed_decisions": ["approve", "reject"]}}),  # HiT-L
        pii_redaction_middleware(patterns=["email", "phone"])  # Scrub PII
    ]
)
```

## Step 5: Custom Middleware (Advanced Hooks)

**Subclass `AgentMiddleware` for `wrap_model_call`, `wrap_tool_call`, etc.**

```python
class DynamicModelMiddleware(AgentMiddleware):
    def wrap_model_call(self, request, handler):
        if len(request.state["messages"]) > 10:  # Complex conv → better model
            request.model = ChatOpenAI(model="gpt-4o")
        return handler(request)

agent = create_agent(model=model, middleware=[DynamicModelMiddleware()])
```

Hooks: `before_model`, `wrap_tool_call`, `after_model`, etc.


**Debug:** Traces auto-sent to LangSmith.

## [Next: LangGraph for Multi-Agent/Complex Flows]

When `create_agent` limits hit (custom edges, subgraphs), migrate to raw LangGraph graphs.

**Relevant docs:**
- [Agents](https://docs.langchain.com/oss/python/langchain/agents)
- [Tools](https://docs.langchain.com/oss/python/langchain/tools)
- [Middleware](https://docs.langchain.com/oss/python/langchain/middleware/built-in)
- [Streaming](https://docs.langchain.com/oss/python/langchain/streaming)

## **Introduction**

Up until now our agents have been powerful but static. Fixed tools, fixed prompt, fixed models.

Middleware let you intercept and customize agent's execution at every step. 

Dynamically swap tools, adjust prompts on the fly, and even change the undelying model based on the situation your agent finds itself in. 

You should be able to summarize, compress and intelligently retain information so that your agent stay coherent over hours or days of interaction without forgetting what matters the most. 

Not every agent action should be left entirely up to an AI system. This is where Human-in-the-loop pattern comes into picture. Add approval checkpoints so that agent and its human counterpart can work together seamlessly.

### **Middleware**
**create_agent()** allows us to build an out of box agent. But it's hard to customize beyond the model, tools and prompts we give it the access to. 

Middleware allows us to insert functions within any interaction inside the agent loop. For example: Let's assume we have a customer support agent that can process refunds. It might not be something that we want to do without human oversight. Here, we can just include a Human-in-the-loop middleware function before the process_refund() tool is called everytime. 

Let's us see how middlewares can be used for:
1. Managing Long Conversations
2. Human in the loop
3. etc...

### **Managing Long Conversations**

With the help of checkpointer, we've got an agent that maintains a list of messages so that it can remember what happened in our conversation till date.

This works well for the shorter conversations. After a while the list of messages become longer, overflowing agents context window. This leads to slower application, and increasing cost. 

There are two ways in middleware we can solve this problem:
1. Summarizing the conversation
2. Trimming or deleting the messages


#### **Summarize Messages**

In [1]:
from langchain_openai import ChatOpenAI

# Setup API Key
f = open('keys/.openai_api_key.txt')
OPENAI_API_KEY = f.read()

openai_chat_model = ChatOpenAI(api_key=OPENAI_API_KEY, 
                               model="gpt-4o-mini", 
                               temperature=1)

In [2]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import SummarizationMiddleware

agent = create_agent(
    model=openai_chat_model,
    checkpointer=InMemorySaver(),
    tools=[],
    middleware=[
        SummarizationMiddleware(
            model=openai_chat_model,
            trigger=("tokens", 100),    # Number of tokens we want our conversation to grow to before summarization
            keep=("messages", 1),    # Number of messages to keep after summarization - it deletes all the previous messages
        )
    ]
)

In [3]:
from langchain.messages import HumanMessage, AIMessage

response = agent.invoke(
    {
        "messages" : [
            HumanMessage(content="What is the population of Zoravia City in the country of Elmandria?"),
            AIMessage(content="As of March 12, 2034, Zoravia City in Elmandria had an estimated population of 18.7 million residents, according to the Arvona Statistical Bureau."),
            
            HumanMessage(content="Who is the current governor of North Velmora State?"),
            AIMessage(content="The current governor of North Velmora State is Helena Droskin, who assumed office on July 4, 2026 after winning the regional assembly vote in Brinfall."),
            
            HumanMessage(content="When was the Grand Solara Bridge in Luneth City constructed?"),
            AIMessage(content="The Grand Solara Bridge in Luneth City was constructed between 1998 and 2003 under the supervision of architect Mateo Krylov from Estavia."),
            
            HumanMessage(content="Which university did Dr. Anika Velor attend before becoming Prime Minister of Torvane?"),
            AIMessage(content="Dr. Anika Velor attended the University of Crestholm in South Argena, graduating in 2001 with a degree in geopolitical engineering before later entering politics in Torvane."),

            HumanMessage(content="What currency is used in the Republic of Virelda, and when was it introduced?"),
        ]
    },
    {
        "configurable" : {"thread_id" : "1"}
    }
)

In [4]:
for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

Here is a summary of the conversation to date:

## SESSION INTENT
The user is seeking factual information about specific locations, government officials, historical constructions, and individuals related to Zoravia City, North Velmora State, Luneth City, and Torvane.

## SUMMARY
- Zoravia City, Elmandria, has an estimated population of 18.7 million residents as of March 12, 2034.
- Helena Droskin is the current governor of North Velmora State, having taken office on July 4, 2026, after winning a regional assembly vote in Brinfall.
- The Grand Solara Bridge in Luneth City was constructed between 1998 and 2003, supervised by architect Mateo Krylov from Estavia.
- Dr. Anika Velor attended the University of Crestholm in South Argena, graduating in 2001 with a degree in geopolitical engineering before entering politics in Torvane.

## ARTIFACTS
None

## NEXT STEPS
None
================================ Human Me

### **Trimming or Deleting Messages**

Run once before or after agent run:
- @before_agent
- @after_agent

Runs multiple times, before or after each model call
- @before_model
- @after_model

#### **Let's say we want to remove all the ToolMessage before the start of an agent run**

Assuming ToolMessages doesn't contain enough information and they are just clutering the context window. 

In [ ]:
from langchain.agents.middleware import before_agent
from langchain.messages import ToolMessage, RemoveMessage

from langgraph.runtime import Runtime

@before_agent
def custom_trim_messages(runtime: Runtime) -> dict | None:
    "Remove all the Tool Message from the state"
    messages = runtime["messages"]

    tool_messages = [msg for msg in messages if isinstance(msg, ToolMessage)]

    return {"messages" : RemoveMessage(id=msg.id) for msg in tool_messages}

### **TODO**

- Foundation Course: https://academy.langchain.com/courses/take/foundation-introduction-to-langchain-python/lessons/71234871-lesson-2-managing-long-conversations
- LangGraph Essentials: https://academy.langchain.com/courses/langgraph-essentials-python
- Introduction to LangGraph (State+Memory, Breakpoints, HITL, LTM and Deployment): https://academy.langchain.com/courses/intro-to-langgraph
- Deep Research with LangGraph (Multi-Agent System): https://academy.langchain.com/courses/deep-research-with-langgraph
- Ambient Agents with LangGraph (Memory and Human in the loop Project) - https://academy.langchain.com/courses/take/ambient-agents/lessons/66147194-agent-evaluations